# Graph-Based Fraud Detection Demo

**Author:** David LECONTE - IBM Worldwide | Data & AI | Tiger Team  
**Date:** 2026-04-11  
**Phase:** 7.4 - Graph-Based Fraud Detection

This notebook demonstrates the complete graph-based fraud detection pipeline including:
- Network analysis and centrality metrics
- Community detection for fraud rings
- Relationship pattern analysis
- Interactive network visualization

## Table of Contents
1. [Setup](#setup)
2. [Data Generation](#data-generation)
3. [Network Analysis](#network-analysis)
4. [Community Detection](#community-detection)
5. [Pattern Analysis](#pattern-analysis)
6. [Visualization](#visualization)
7. [Investigation Workflow](#investigation-workflow)
8. [Conclusions](#conclusions)

## 1. Setup <a name="setup"></a>

Import required modules and configure environment.

In [ ]:
# Standard library imports
import sys
from pathlib import Path
from typing import Dict, List, Set

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Graph analysis modules
from banking.graph import (
    NetworkAnalyzer,
    CommunityDetector,
    PatternAnalyzer,
    GraphVisualizer,
    VisualizationConfig,
)

# Identity generation
from banking.identity import SyntheticIdentityGenerator

# Visualization
import networkx as nx
import pandas as pd

print("✅ Modules imported successfully")

## 2. Data Generation <a name="data-generation"></a>

Generate synthetic identities for fraud detection analysis.

In [ ]:
# Generate synthetic identities
print("Generating synthetic identities...")
generator = SyntheticIdentityGenerator(seed=42)
identities = generator.generate_batch(50)

print(f"✅ Generated {len(identities)} identities")
print(f"\nSample identity:")
sample = identities[0]
for key, value in list(sample.items())[:8]:
    print(f"  {key}: {value}")

## 3. Network Analysis <a name="network-analysis"></a>

Build network from identities and calculate centrality metrics.

In [ ]:
# Build network
print("Building network from identities...")
network_analyzer = NetworkAnalyzer()
G = network_analyzer.build_network(identities)

print(f"✅ Network built:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")

In [ ]:
# Calculate centrality metrics
print("Calculating centrality metrics...")
centrality_metrics = network_analyzer.calculate_centrality(G)

# Extract risk scores
risk_scores = {
    node_id: metrics.get_risk_score()
    for node_id, metrics in centrality_metrics.items()
}

# Create DataFrame for analysis
centrality_df = pd.DataFrame([
    {
        'node_id': node_id,
        'degree': metrics.degree_centrality,
        'betweenness': metrics.betweenness_centrality,
        'closeness': metrics.closeness_centrality,
        'pagerank': metrics.pagerank,
        'risk_score': metrics.get_risk_score(),
        'role': metrics.get_role(),
    }
    for node_id, metrics in centrality_metrics.items()
]).sort_values('risk_score', ascending=False)

print(f"\n✅ Centrality calculated for {len(centrality_metrics)} nodes")
print(f"\nTop 10 High-Risk Nodes:")
print(centrality_df.head(10).to_string(index=False))

In [ ]:
# Calculate network metrics
print("Calculating network metrics...")
network_metrics = network_analyzer.get_network_metrics(G)

print(f"\n✅ Network Metrics:")
print(f"  Node Count: {network_metrics.node_count}")
print(f"  Edge Count: {network_metrics.edge_count}")
print(f"  Density: {network_metrics.density:.3f}")
print(f"  Avg Clustering: {network_metrics.average_clustering:.3f}")
print(f"  Diameter: {network_metrics.diameter}")
print(f"  Avg Path Length: {network_metrics.average_path_length:.2f}")
print(f"  Connected Components: {network_metrics.connected_components}")
print(f"  Largest Component: {network_metrics.largest_component_size}")
print(f"  Network Risk Score: {network_metrics.get_network_risk_score():.1f}/100")

## 4. Community Detection <a name="community-detection"></a>

Detect communities and identify potential fraud rings.

In [ ]:
# Detect communities
print("Detecting communities...")
community_detector = CommunityDetector()
community_result = community_detector.detect_communities(G, algorithm="louvain")

print(f"\n✅ Community Detection Results:")
print(f"  Communities: {community_result.num_communities}")
print(f"  Modularity: {community_result.modularity:.3f}")
print(f"  Algorithm: {community_result.algorithm}")

# Analyze communities
print(f"\nCommunity Statistics:")
for comm_id, community in sorted(community_result.communities.items()):
    print(f"  Community {comm_id}:")
    print(f"    Size: {community.size}")
    print(f"    Density: {community.density:.3f}")
    print(f"    Risk Score: {community.get_risk_score():.1f}")
    print(f"    Risk Level: {community.get_risk_level().upper()}")

In [ ]:
# Identify fraud rings
print("Identifying fraud rings...")
fraud_rings = community_result.get_fraud_rings(min_size=3, min_risk=60.0)

print(f"\n✅ Found {len(fraud_rings)} potential fraud rings:")
for i, ring in enumerate(fraud_rings, 1):
    print(f"\nFraud Ring {i}:")
    print(f"  Size: {ring.size} members")
    print(f"  Density: {ring.density:.3f}")
    print(f"  Risk Score: {ring.get_risk_score():.1f}/100")
    print(f"  Risk Level: {ring.get_risk_level().upper()}")
    print(f"  Members: {', '.join(list(ring.members)[:5])}...")

## 5. Pattern Analysis <a name="pattern-analysis"></a>

Analyze relationship patterns including shared attributes, circular patterns, and velocity.

In [ ]:
# Analyze patterns
print("Analyzing relationship patterns...")
pattern_analyzer = PatternAnalyzer()
pattern_result = pattern_analyzer.analyze_patterns(G, identities)

# Get summary
summary = pattern_result.get_pattern_summary()

print(f"\n✅ Pattern Analysis Results:")
print(f"  Shared Attribute Patterns: {summary['shared_attribute_patterns']}")
print(f"  Circular Patterns: {summary['circular_patterns']}")
print(f"  Layering Patterns: {summary['layering_patterns']}")
print(f"  Velocity Patterns: {summary['velocity_patterns']}")
print(f"  Total Patterns: {summary['total_patterns']}")

In [ ]:
# Analyze shared attribute patterns
if pattern_result.shared_attribute_patterns:
    print("\nTop Shared Attribute Patterns:")
    for i, pattern in enumerate(pattern_result.shared_attribute_patterns[:5], 1):
        print(f"\n{i}. {pattern.attribute_type.upper()} Pattern:")
        print(f"   Entities: {pattern.entity_count}")
        print(f"   Risk Score: {pattern.get_risk_score():.1f}/100")
        print(f"   Risk Level: {pattern.get_risk_level().upper()}")
else:
    print("\nNo shared attribute patterns detected")

In [ ]:
# Analyze circular patterns
if pattern_result.circular_patterns:
    print("\nTop Circular Patterns:")
    for i, pattern in enumerate(pattern_result.circular_patterns[:5], 1):
        print(f"\n{i}. Cycle Length {pattern.length}:")
        print(f"   Path: {' → '.join(pattern.cycle[:4])}...")
        print(f"   Risk Score: {pattern.get_risk_score():.1f}/100")
        print(f"   Risk Level: {pattern.get_risk_level().upper()}")
else:
    print("\nNo circular patterns detected")

In [ ]:
# Get high-risk patterns
high_risk_patterns = pattern_result.get_high_risk_patterns(min_risk=60.0)

print("\nHigh-Risk Patterns (≥60):")
print(f"  Shared Attributes: {len(high_risk_patterns['shared_attributes'])}")
print(f"  Circular: {len(high_risk_patterns['circular'])}")
print(f"  Layering: {len(high_risk_patterns['layering'])}")
print(f"  Velocity: {len(high_risk_patterns['velocity'])}")

## 6. Visualization <a name="visualization"></a>

Create interactive network visualizations.

In [ ]:
# Create visualizer
print("Creating network visualization...")
config = VisualizationConfig(
    layout="spring",
    width=1200,
    height=800,
    show_labels=True,
    enable_hover=True,
)
visualizer = GraphVisualizer(config)

try:
    # Create visualization with risk scores
    fig = visualizer.visualize(
        G,
        risk_scores=risk_scores,
        fraud_rings=[ring.members for ring in fraud_rings],
        title="Fraud Network Analysis",
        output_file=Path("../output/fraud_network.html"),
    )
    
    print("✅ Visualization created: ../output/fraud_network.html")
    print("   Open in browser to view interactive network")
    
    # Display in notebook if plotly available
    if fig is not None:
        fig.show()
except ImportError as e:
    print(f"⚠️  Visualization requires plotly: {e}")
    print("   Install with: pip install plotly")

## 7. Investigation Workflow <a name="investigation-workflow"></a>

Complete investigation workflow with prioritization.

In [ ]:
# Identify investigation targets
print("Identifying investigation targets...")

# High-risk nodes
high_risk_nodes = [n for n, s in risk_scores.items() if s >= 60.0]

# Fraud ring members
fraud_ring_members = set()
for ring in fraud_rings:
    fraud_ring_members.update(ring.members)

# High-risk pattern entities
pattern_entities = set()
for pattern in high_risk_patterns['shared_attributes']:
    pattern_entities.update(pattern.entities)

# Combine all targets
all_targets = set(high_risk_nodes) | fraud_ring_members | pattern_entities

print(f"\n✅ Investigation Targets Identified:")
print(f"  High-Risk Nodes: {len(high_risk_nodes)}")
print(f"  Fraud Ring Members: {len(fraud_ring_members)}")
print(f"  Pattern Entities: {len(pattern_entities)}")
print(f"  Total Unique Targets: {len(all_targets)}")

In [ ]:
# Generate investigation report
print("\n" + "=" * 80)
print("FRAUD INVESTIGATION REPORT")
print("=" * 80)

print(f"\nDATA SUMMARY:")
print(f"  Total Identities: {len(identities)}")
print(f"  Network Nodes: {G.number_of_nodes()}")
print(f"  Network Edges: {G.number_of_edges()}")
print(f"  Network Density: {nx.density(G):.3f}")

print(f"\nCOMMUNITY ANALYSIS:")
print(f"  Communities Detected: {community_result.num_communities}")
print(f"  Modularity Score: {community_result.modularity:.3f}")
print(f"  Fraud Rings Identified: {len(fraud_rings)}")

print(f"\nPATTERN ANALYSIS:")
print(f"  Total Patterns: {summary['total_patterns']}")
print(f"  High-Risk Patterns: {sum(len(p) for p in high_risk_patterns.values())}")

print(f"\nRISK ASSESSMENT:")
print(f"  High-Risk Nodes (≥60): {len(high_risk_nodes)}")
print(f"  Network Risk Score: {network_metrics.get_network_risk_score():.1f}/100")

print(f"\nINVESTIGATION TARGETS:")
print(f"  Priority Targets: {len(all_targets)}")
print(f"  Coverage: {len(all_targets)/len(identities)*100:.1f}% of identities")

print(f"\nRECOMMENDED ACTIONS:")
if len(fraud_rings) >= 3:
    print("  🚨 URGENT: Multiple fraud rings detected")
    print("  - Escalate to fraud investigation team immediately")
    print("  - Freeze suspicious accounts")
    print("  - Initiate detailed forensic analysis")
elif len(fraud_rings) >= 1:
    print("  ⚠️  HIGH: Fraud rings detected")
    print("  - Conduct detailed review of identified rings")
    print("  - Monitor transactions closely")
    print("  - Prepare compliance documentation")
else:
    print("  ℹ️  MEDIUM: Patterns detected")
    print("  - Continue routine monitoring")
    print("  - Review high-risk nodes")
    print("  - Update risk models")

print("\n" + "=" * 80)

## 8. Conclusions <a name="conclusions"></a>

### Key Findings

1. **Network Structure**: The fraud network exhibits characteristics typical of organized fraud:
   - Multiple connected components
   - High clustering in fraud rings
   - Clear hub nodes (coordinators)

2. **Fraud Rings**: Community detection successfully identified organized fraud groups:
   - Tight-knit communities with high internal density
   - Clear separation from legitimate accounts
   - Quantifiable risk scores for prioritization

3. **Pattern Analysis**: Multiple suspicious patterns detected:
   - Shared attributes indicating synthetic identities
   - Circular transaction patterns suggesting money laundering
   - Rapid connection formation indicating account takeover

4. **Risk Assessment**: Comprehensive risk scoring enables:
   - Prioritized investigation workflows
   - Resource allocation optimization
   - Compliance documentation

### Business Value

- **Detection**: Automated identification of fraud networks
- **Investigation**: Visual and quantitative evidence for investigators
- **Compliance**: Audit-ready documentation and reporting
- **Prevention**: Early warning system for emerging fraud patterns

### Next Steps

1. Deploy to production environment
2. Integrate with real-time transaction monitoring
3. Establish automated alerting thresholds
4. Train investigation team on tools
5. Implement continuous model improvement

---

**Prepared by:** David LECONTE - IBM Worldwide | Data & AI | Tiger Team  
**Date:** 2026-04-11  
**Phase:** 7.4 - Graph-Based Fraud Detection